In [9]:
import pandas as pd
import numpy as np
import joblib

In [10]:
model = joblib.load("disease_prediction_naive_bayes.pkl")
label_encoder = joblib.load("disease_label_encoder.pkl")
scaler = joblib.load("disease_scaler.pkl")
feature_columns = joblib.load("disease_feature_columns.pkl")
numerical_columns = joblib.load("disease_numerical_columns.pkl")

C:\Users\ANANYA\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator GaussianNB from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\ANANYA\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\ANANYA\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator 

In [11]:
print("Model:", type(model))
print("Number of features:", len(feature_columns))
print("Numerical columns:", numerical_columns)
print("Disease classes:", label_encoder.classes_)

Model: <class 'sklearn.naive_bayes.GaussianNB'>
Number of features: 41
Numerical columns: ['Age', 'Blood_Pressure', 'Sugar_Level', 'Cholesterol']
Disease classes: ['Diabetes' 'Flu' 'Gastroenteritis' 'Healthy' 'Heart Disease'
 'Hypertension' 'Migraine' 'Pneumonia']


In [12]:

print("Total features:", len(feature_columns))

print("\nFeatures expected by the model:")
for i, feature in enumerate(feature_columns, start=1):
    print(i, "→", feature)

Total features: 41

Features expected by the model:
1 → Age
2 → Blood_Pressure
3 → Sugar_Level
4 → Cholesterol
5 → Unknown
6 → abdominal pain
7 → blurred vision
8 → body ache
9 → chest pain
10 → cough
11 → diarrhea
12 → dizziness
13 → fatigue
14 → fever
15 → frequent urination
16 → headache
17 → increased thirst
18 → nausea
19 → sensitivity to light
20 → shortness of breath
21 → sore throat
22 → vomiting
23 → Gender_Male
24 → Medical_History_Diabetes
25 → Medical_History_Family history of diabetes
26 → Medical_History_Family history of heart disease
27 → Medical_History_Family history of hypertension
28 → Medical_History_Family history of migraine
29 → Medical_History_Gastritis
30 → Medical_History_Heart disease
31 → Medical_History_High cholesterol
32 → Medical_History_Hypertension
33 → Medical_History_Migraine
34 → Medical_History_Previous gastrointestinal infection
35 → Medical_History_Previous respiratory infection
36 → Medical_History_Seasonal flu
37 → Medical_History_Unknown
38 →

In [13]:
print("Numerical columns:")
print(numerical_columns)

Numerical columns:
['Age', 'Blood_Pressure', 'Sugar_Level', 'Cholesterol']


In [14]:
patient = pd.DataFrame({
    "Age": [45],
    "Gender": ["Male"],
    "Symptoms": ["Fever, Cough"],
    "Blood_Pressure": [140],
    "Sugar_Level": [160],
    "Cholesterol": [220],
    "Medical_History": ["Diabetes"]
})

patient

,Age,Gender,Symptoms,Blood_Pressure,Sugar_Level,Cholesterol,Medical_History
0,45,Male,"Fever, Cough",140,160,220,Diabetes


In [15]:
print("Patient shape:", patient.shape)
print("\nPatient columns:")
print(patient.columns.tolist())

Patient shape: (1, 7)

Patient columns:
['Age', 'Gender', 'Symptoms', 'Blood_Pressure', 'Sugar_Level', 'Cholesterol', 'Medical_History']


In [16]:
all_symptoms = [
    "Unknown",
    "abdominal pain",
    "blurred vision",
    "body ache",
    "chest pain",
    "cough",
    "diarrhea",
    "dizziness",
    "fatigue",
    "fever",
    "frequent urination",
    "headache",
    "increased thirst",
    "nausea",
    "sensitivity to light",
    "shortness of breath",
    "sore throat",
    "vomiting"
]

In [17]:
for symptom in all_symptoms:
    patient[symptom] = patient["Symptoms"].apply(
        lambda x: 1 if symptom in [s.strip() for s in x.split(",")] else 0
    )

In [18]:
patient = patient.drop("Symptoms", axis=1)

In [19]:
patient.head()

,Age,Gender,Blood_Pressure,Sugar_Level,Cholesterol,Medical_History,Unknown,abdominal pain,blurred vision,body ache,...,fatigue,fever,frequent urination,headache,increased thirst,nausea,sensitivity to light,shortness of breath,sore throat,vomiting
0,45,Male,140,160,220,Diabetes,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
def get_age_group(age):
    if 18 <= age <= 30:
        return "18-30"
    elif 31 <= age <= 45:
        return "31-45"
    elif 46 <= age <= 60:
        return "46-60"
    elif 61 <= age <= 75:
        return "61-75"
    elif age >= 76:
        return "76+"
    else:
        return "Unknown"

In [21]:
patient["Age_Group"] = patient["Age"].apply(get_age_group)

In [22]:
print(patient[["Age", "Age_Group"]])

   Age Age_Group
0   45     31-45


In [23]:
categorical_columns = [
    "Gender",
    "Medical_History",
    "Age_Group"
]

patient = pd.get_dummies(
    patient,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

In [24]:
print("Patient features before alignment:")
print(patient.columns.tolist())

print("\nNumber of columns:", len(patient.columns))

Patient features before alignment:
['Age', 'Blood_Pressure', 'Sugar_Level', 'Cholesterol', 'Unknown', 'abdominal pain', 'blurred vision', 'body ache', 'chest pain', 'cough', 'diarrhea', 'dizziness', 'fatigue', 'fever', 'frequent urination', 'headache', 'increased thirst', 'nausea', 'sensitivity to light', 'shortness of breath', 'sore throat', 'vomiting']

Number of columns: 22


In [25]:
patient = patient.reindex(
    columns=feature_columns,
    fill_value=0
)

In [26]:
print("Final patient shape:", patient.shape)
print("Expected shape: (1, 41)")

Final patient shape: (1, 41)
Expected shape: (1, 41)


In [27]:
print(patient)

   Age  Blood_Pressure  Sugar_Level  Cholesterol  Unknown  abdominal pain  \
0   45             140          160          220        0               0   

   blurred vision  body ache  chest pain  cough  ...  \
0               0          0           0      0  ...   

   Medical_History_Hypertension  Medical_History_Migraine  \
0                             0                         0   

   Medical_History_Previous gastrointestinal infection  \
0                                                  0     

   Medical_History_Previous respiratory infection  \
0                                               0   

   Medical_History_Seasonal flu  Medical_History_Unknown  Age_Group_31-45  \
0                             0                        0                0   

   Age_Group_46-60  Age_Group_61-75  Age_Group_76+  
0                0                0              0  

[1 rows x 41 columns]


In [28]:
print("\nFinal feature count:", patient.shape[1])


Final feature count: 41


In [30]:
patient[numerical_columns] = scaler.transform(
    patient[numerical_columns]
)

In [31]:
print(patient.shape)
print(patient[numerical_columns])

(1, 41)
        Age  Blood_Pressure  Sugar_Level  Cholesterol
0 -0.086774        0.556261     1.353801     0.316934


In [32]:
prediction = model.predict(patient)

print("Encoded prediction:", prediction)

Encoded prediction: [5]


In [33]:
predicted_disease = label_encoder.inverse_transform(prediction)

print("Predicted Disease:", predicted_disease[0])

Predicted Disease: Hypertension


In [34]:
probabilities = model.predict_proba(patient)

max_probability = np.max(probabilities)

print("Prediction Probability:", round(max_probability * 100, 2), "%")

Prediction Probability: 62.14 %


In [35]:
print("================================")
print("FINAL DISEASE PREDICTION")
print("================================")
print("Disease:", predicted_disease[0])
print("Confidence:", round(max_probability * 100, 2), "%")

FINAL DISEASE PREDICTION
Disease: Hypertension
Confidence: 62.14 %


In [36]:
def predict_disease(patient):
    patient = patient.copy()

    # Symptom features
    for symptom in all_symptoms:
        patient[symptom] = patient["Symptoms"].apply(
            lambda x: 1 if symptom in [s.strip() for s in x.split(",")] else 0
        )

    patient = patient.drop("Symptoms", axis=1)

    # Age group
    patient["Age_Group"] = patient["Age"].apply(get_age_group)

    # Encoding
    patient = pd.get_dummies(
        patient,
        columns=["Gender", "Medical_History", "Age_Group"],
        drop_first=True,
        dtype=int
    )

    # Exact 41 features
    patient = patient.reindex(
        columns=feature_columns,
        fill_value=0
    )

    # Scaling
    patient[numerical_columns] = scaler.transform(
        patient[numerical_columns]
    )

    # Prediction
    prediction = model.predict(patient)
    disease = label_encoder.inverse_transform(prediction)[0]

    probability = np.max(model.predict_proba(patient)) * 100

    return disease, probability

In [37]:
disease, confidence = predict_disease(
    pd.DataFrame({
        "Age": [45],
        "Gender": ["Male"],
        "Symptoms": ["Fever, Cough"],
        "Blood_Pressure": [140],
        "Sugar_Level": [160],
        "Cholesterol": [220],
        "Medical_History": ["Diabetes"]
    })
)

print("Disease:", disease)
print("Confidence:", round(confidence, 2), "%")

Disease: Hypertension
Confidence: 62.14 %
